# Report — Hydropower Unit Commitment (Tokke–Vinje)

**Course:** TDT4173 Modern Machine Learning in Practice — Course Project
**Group number:** `[TODO]`
**Kaggle team name:** `[TODO — must include the group number, e.g. "[34] A Fantastic Team"]`

| Full name | Student ID (from student card) |
|-----------|-------------------------------|
| `[TODO]`  | `[TODO]` |
| `[TODO]`  | `[TODO]` |
| `[TODO]`  | `[TODO]` |

Full write-up of our group work. Clearly titled sections below map directly to the graded items (EDA, ≥2 predictors, feature engineering, model interpretation).


## 0. Executive summary & results

One paragraph: the task, our best model, CV score, public/private LB, and which 2 Kaggle predictions we selected. Fill in the final numbers before submitting.

> **Graders:** this report is **not run** — all outputs below are saved from our actual runs.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 1. Problem understanding & domain knowledge  *(EDA item: domain knowledge / how the data was generated)*

> **Steps for this section (temporary — remove before submission)**
> - What hydropower **unit commitment** is and why ON/OFF per unit per hour matters.
> - How the data was generated: SHOP v17.17.0 optimization on the Tokke–Vinje watercourse; each case is a 168-hour rolling-horizon week.
> - The economic driver: generate when price > marginal water value, subject to inflow, storage, min-flow/min-volume constraints and the cascade topology.
>
> *Notes on the steps:* Steps 1 and 3 hold. Step 3 is only the first-order rule, so 1.3 adds the refinement that matters most for modelling: the price's *rank* within the week. Step 2 needs a correction. "Rolling horizon" usually means re-optimising and keeping only the first day. Here each case is a full, separate 168 h solve, and a new case starts every day. So every hour of every case is a label, and neighbouring cases share 6 of their 7 days, which matters for validation (H6). Step 2 should also say that the labels come from an optimiser, not from operators.

### 1.1 Unit commitment
For each unit and hour, unit commitment decides whether the unit is **ON** (producing at least its minimum stable output `p_min`) or **OFF**. How much a running unit produces is a separate decision. ON/OFF is a real choice and not just "produce zero", for two reasons. First, `p_min` ≈ 34 % of `p_max` for most units (table below). Second, starts and stops cost money and cause wear, so schedules come in continuous blocks. We predict P(ON) for 14 units × 168 hours = 2352 unit-hours per case, scored with micro-averaged ROC-AUC.

### 1.2 How the data was generated
The labels are **not human decisions**. They are optimal schedules from SINTEF's short-term scheduling optimiser **SHOP v17.17.0** on the Tokke–Vinje model, which has 17 reservoirs, 9 plants, 14 units and ≈ 1 GW of capacity. Each case (`Run No`) is one deterministic 168 h optimisation starting at `starttime`, with these inputs:

- **hourly price and inflow**, assumed known for the whole week;
- **initial reservoir volumes** (daily data);
- **water values** at `starttime + 7 days`, i.e. what water left at the end of the week is worth;
- **min-flow and min-volume constraints** (rarely active);
- the **topology**: connections, efficiencies, `p_min`/`p_max`.

A new case starts every day (output below), so consecutive cases overlap by 144 of 168 hours.

The system is a **cascade**: Kjela / Vesle Kjela / Songa → Totak → **Vinje** (3 × 100 MW) → **Tokke** (4 × 107.5 MW) → Bandak → Hogga → out. Byrte → Lio and Haukeli feed in from the side. The two largest plants therefore use water released upstream, so the units are not independent.

In principle the targets are a function of the inputs, so the ML task is to learn a fast **surrogate of SHOP**. There are two caveats. The water values are *synthetic*, and the Hyljelihyl and Vatjern volumes are fixed at 50 % because no records exist, so those two series carry no information.

### 1.3 The economic driver: price vs. water value
The **water value** is the opportunity cost of using water now instead of later. To a first approximation, a unit runs in hour *t* when the price is above the water value of the water it uses. Prices are known for the whole week and water is limited, so SHOP puts production into the **highest-price hours**. The **rank of an hour's price within its week** therefore matters more than its absolute level. Other factors shift this rule:

- **minimum load and start costs** make units run in blocks;
- **storage and inflow** matter: a nearly full reservoir with high inflow is drained even at low prices to avoid spilling, while a nearly empty one saves water;
- **constraints** can force a unit ON or OFF;
- the **cascade** links downstream plants to upstream releases;
- **identical units** (Tokke G1–G4, Vinje G1–G3) are interchangeable, so which of them SHOP picks is partly arbitrary. This puts a ceiling on how well a single unit can be predicted.

The code below extracts these system facts from the raw data.

In [1]:
from pathlib import Path
import yaml
import pandas as pd

DATA_DIR = Path("sintef_project/data")   # same config constant as the short notebooks

topo = yaml.safe_load(open(DATA_DIR / "extended" / "Tokke_Vinje_topology.yaml", encoding="utf-8"))
model, conns = topo["model"], topo["connections"]

# --- Units: capacity limits and where each plant discharges its water ---
outlet = {c["from"]: c["to"] for c in conns if c["from_type"] == "plant"}
units = pd.DataFrame([
    {"unit": g, "plant": g.rsplit("_", 1)[0], "p_min [MW]": v["p_min"], "p_max [MW]": v["p_max"]}
    for g, v in model["generator"].items()
])
units["p_min / p_max"] = (units["p_min [MW]"] / units["p_max [MW]"]).round(2)
units["discharges into"] = units["plant"].map(outlet).fillna("(leaves system)")
display(units.set_index("unit"))
print(f"Plants: {units.plant.nunique()}, units: {len(units)}, "
      f"total installed capacity: {units['p_max [MW]'].sum():.1f} MW")

# --- Reservoirs: size (max_vol) ---
res = (pd.Series({r: v["max_vol"] for r, v in model["reservoir"].items()}, name="max_vol [Mm³]")
         .sort_values(ascending=False))
display(res.to_frame().T)

# --- Case structure: rolling weekly optimisations ---
uc = pd.read_csv(DATA_DIR / "kernel" / "Unit_commitment_decisions.csv",
                 usecols=["Run No", "starttime", "total_calculation_time"], parse_dates=["starttime"])
print(f"Training cases: {len(uc)} (Run No {uc['Run No'].min()}–{uc['Run No'].max()}), "
      f"{uc.starttime.min().date()} → {uc.starttime.max().date()}")
print("Days between consecutive case starts:", uc.starttime.diff().dt.days.value_counts().to_dict())
print("Each case = 168 h → consecutive cases overlap by 144 h (6 of 7 days)")
print("SHOP solve time per case [s]:",
      uc.total_calculation_time.describe()[["mean", "min", "max"]].round(1).to_dict())

,plant,p_min [MW],p_max [MW],p_min / p_max,discharges into
unit,,,,,
Hogga_G1,Hogga,3.0,17.0,0.18,(leaves system)
Tokke_G1,Tokke,36.9,107.5,0.34,Bandak
Tokke_G2,Tokke,36.9,107.5,0.34,Bandak
Tokke_G3,Tokke,36.9,107.5,0.34,Bandak
Tokke_G4,Tokke,36.9,107.5,0.34,Bandak
Lio_G1,Lio,14.8,43.0,0.34,r_Vest_Vassdraget
Byrte_G1,Byrte,6.9,20.0,0.34,Byrtevatn
Vinje_G1,Vinje,34.4,100.0,0.34,Vinjevatn
Vinje_G2,Vinje,34.4,100.0,0.34,Vinjevatn


Plants: 9, units: 14, total installed capacity: 1003.4 MW


,Songavatn,Totak,Bordalsvatn,Bandak,Foersvatn,Bitdalsvatn,Kjelavatn,Langesae,Byrtevatn,Botnedalsvatn,Staavatn,Langeidvatn,Vaamarvatn,Venemo,Vinjevatn,Vatjern,Hyljelihyl
max_vol [Mm³],640.0,257.8,184.2,174.2,122.3,110.0,84.5,78.0,76.71,54.17,48.0,32.1,26.2,23.0,9.36,0.3,0.26


Training cases: 2922 (Run No 1–2922), 2015-01-01 → 2022-12-31
Days between consecutive case starts: {1.0: 2921}
Each case = 168 h → consecutive cases overlap by 144 h (6 of 7 days)
SHOP solve time per case [s]: {'mean': 147.0, 'min': 22.4, 'max': 4525.7}


### 1.4 Hypotheses for later sections

| # | Hypothesis | Checked in |
|---|---|---|
| H1 | P(ON) rises with the **within-week price rank** | EDA 2.3, PDP (Sec. 6) |
| H2 | **Price − water value** is predictive | EDA 2.3, importance (Sec. 6) |
| H3 | High **fill fraction / inflow** raise ON-rates (spill avoidance) | EDA 2.3 |
| H4 | Strong **hour-of-day** and **seasonal** patterns | EDA 2.2 |
| H5 | Units at the same plant are **co-activated**; identical units may differ in ON-rate | EDA 2.3 |
| H6 | Overlapping cases make **random K-fold leak**, so we use a **time-based split** | Sec. 5 |

**Pipeline consequences:** use *relative* price features (rank and deviation within the case), use static topology (capacity, plant, cascade position) as unit features, and drop the constant Hyljelihyl/Vatjern volume columns.

## 2. Exploratory Data Analysis  *(covers ≥4 EDA items → avoids −3)*

> **Steps for this section (temporary — remove before submission)**
> - **2.1 Data sanity / is it intuitive** — timestamp coverage & UTC alignment, daily vs hourly granularity, the Hyljelihyl/Vatjern 50%-storage assumption, missing values.
> - **2.2 Individual features** — price distribution & seasonality; per-unit ON-rate; ON-rate by hour-of-day and season.
> - **2.3 Pairs / groups of features** — price vs commitment; water value vs commitment; correlations between reservoirs; unit co-activation within a plant.
> - **2.4 Feature cleanup** — timezone handling, joining daily series onto hourly rows, constant/near-constant constraint columns.

### 2.1 Data sanity
All time series are stamped in **UTC**. The hourly series are price, inflow and the two constraint files. The daily series (volume and water value) are stamped at 00:00 UTC, and every case starts at 00:00 UTC. A case's 168 hours can therefore be joined to the hourly data with `starttime + hour`, and to the daily data by date, without time-zone shifts. The data covers the whole test period: the last test case starts on 2024-12-25 and needs prices up to 2024-12-31 23:00 and the water value for 2025-01-01. The volumes of **Hyljelihyl and Vatjern** stay at about 50 % of `max_vol` throughout, which confirms the dataset note that they are filled-in values and not measurements. `[TODO: missing-value counts per table]`

### 2.2 Individual features and the target
- **Price** `[TODO: figure — weekly mean price 2015–2024, test period shaded]`. Prices are low and stable in 2015–2020, rise sharply in 2021–2022 during the energy crisis, and are lower again in the 2023–2024 test period. The test distribution is therefore unlike the most recent training year, which is one reason we use *relative* price features (Section 3).
- **Target.** The overall ON-rate is **0.516**, so the classes are balanced. ON-rates differ a lot between units `[TODO: figure — ON-rate per unit]`.
- **Hour of day and season** `[TODO: figure — ON-rate heatmaps unit × hour-of-day and unit × month]`. We expect commitment to follow the daily price pattern (more often ON in the day and evening peaks than at night) and a seasonal pattern linked to the spring flood and winter prices (H4).
- **Schedules come in blocks.** Units switch ON/OFF only a few times per week, matching the minimum load and start costs described in Section 1. `[TODO: switches per unit-week, share of weeks fully ON/OFF]`

### 2.3 Pairs and groups of features
- **Price vs commitment (H1).** ON-rate against the decile of the hour's price within its week, per plant `[TODO: figure]`.
- **Price vs water value (H2).** ON-rate against the spread between price and the unit's net water value; the economic rule predicts a sharp rise around zero `[TODO: figure]`.
- **Reservoir state (H3).** ON-rate against fill fraction and weekly inflow of the intake reservoir; this partly overlaps with season.
- **Reservoir correlation.** Water values are expected to be strongly correlated across reservoirs, since they are derived from the same market outlook; fill fractions less so.
- **Co-activation (H5).** Correlation of ON/OFF between units; units at the same plant and along the same chain should be strongly positively correlated `[TODO: figure — unit correlation matrix]`.
- **Overlapping cases (H6).** Neighbouring cases share 144 of their 168 hours, so they are near-duplicates and random K-fold would leak. The same calendar hour can still get different labels in two overlapping cases, mostly near the end of the earlier case's horizon, where the end-of-week water value takes over.

### 2.4 Feature cleanup
- Time zones: the UTC offset is removed from every timestamp, and the daily series are normalised to midnight before joining.
- Daily data is joined with an as-of lookup: volume and water value at the case start date, and water value at start + 7 days for the end of the week.
- Hyljelihyl and Vatjern volumes are constant and carry no information.
- The min-volume constraints are zero almost all of the time, and several min-flow series take only a few distinct values. They are kept as raw columns, but are candidates for the near-constant filter in Section 3.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 3. Feature Engineering  *(avoids −3)*

> **Steps for this section (temporary — remove before submission)**
> - Extraction: price rank within the week, rolling price stats, water-value spread (start vs end-of-week), reservoir fill fraction (volume / max_vol from topology), inflow aggregates, upstream/cascade features, unit & plant identity, historical ON-rate priors, calendar features.
> - Selection: importance/permutation-based pruning, correlation filter, near-constant removal — report what was dropped and the effect on CV.
>
> *Code: Short notebook 1, Sections 3 and 3b.*

### 3.1 Framing
We model a **long table** with one row per (case, unit, hour) and the target ON/OFF. That gives 2922 × 14 × 168 ≈ **6.9 M training rows** and 1.7 M test rows. One binary classifier predicts P(ON) for every row, which matches the micro-averaged ROC-AUC metric directly. Predictions are pivoted back to the wide 2352-column format at the end.

### 3.2 Feature extraction
The feature groups are chosen from the domain analysis in Section 1, for 81 features in total:

| Group | Examples | Why |
|---|---|---|
| Unit & topology | unit/plant id, `p_min`, `p_max`, start cost, units at plant, intake/outlet reservoir | units differ in size and cascade position |
| Calendar | hour of day, weekday, month, cyclic sin/cos, hour in horizon | daily and seasonal patterns (H4), end-of-horizon effect |
| Relative price | rank of the hour's price in the week and in the day, deviation and z-score vs week mean, lags/leads, forward 3/6/12 h mean and max | SHOP knows all prices of the week and fills the most expensive hours first (H1) |
| Water value & storage | price − water value (start, end, *net* = intake minus outlet value), price/water-value ratio, fill fraction, weekly inflow relative to reservoir size, margin to min volume, downstream reservoir state, system means | the economic rule and spill avoidance (H2, H3) |
| "In the money" | share of the week's hours with price > water value, and how far the hour's rank lies above that cut-off | turns the rule into an estimate of how many hours the unit should run |
| Constraints | hourly min-flow / min-volume per restricted river/reservoir | can force units ON |
| ON-rate priors | historical P(ON) per unit × hour-of-day and unit × month, computed on the training fold only | strong baseline, leak-free |
| **LP features (simplified SHOP)** | for each case, solve a linear relaxation of SHOP's weekly problem (water balance through the topology, reservoir and discharge limits, constraints, end-of-week water values) and use its plant discharge, the unit's implied load and the price minus the LP's marginal water cost | lets the model see the optimiser's own trade-off |

The LP features are the most important extraction step. **Used alone as a score, without any learning, the LP margin already gives ROC-AUC 0.956 on the training set.** The LP leaves out the ON/OFF integers, start costs, `p_min` and head variation; the gradient-boosting model learns to correct for those.

### 3.3 Feature selection
- **Near-constant and duplicate columns.** Constant reservoir series are excluded (Section 2.4). `hour` and `hour_in_horizon` are identical, so one is redundant. `[TODO: list of near-constant / |ρ| > 0.98 columns dropped and the resulting CV]`
- **Group ablation (before the LP features).** With LightGBM on three rolling yearly folds (validation years 2020, 2021, 2022, each with a 7-day purge gap, every 4th case), we tested adding and removing feature groups:

| Variant | 2020 | 2021 | 2022 | Mean |
|---|---|---|---|---|
| Baseline (73 features) | 0.9406 | 0.9488 | 0.9614 | 0.9503 |
| + water-budget features | 0.9452 | 0.9483 | 0.9612 | 0.9516 |
| − static unit features | 0.9450 | 0.9452 | 0.9632 | 0.9511 |
| − absolute price | 0.9441 | 0.9462 | 0.9623 | 0.9509 |
| + all 22 candidate features | 0.9419 | 0.9488 | 0.9611 | 0.9506 |

Repeating runs with three seeds changed the 2020 score by up to ±0.003, so none of these differences is larger than seed noise. We therefore did **not** add the candidate groups (end-of-week volume, daily release, block features, unit order), which keeps the feature set smaller.

- **Effect of the LP features.** Adding the LP features raised the time-split validation AUC from roughly 0.96 to **0.990** (Short notebook 1). This is by far the largest single improvement.
- Gain importance confirms this: the top six features are `lp_unit_load`, `lp_qfrac`, `lp_margin_ctr5`, `unit_id`, `lp_hours_pos_frac` and `lp_margin` (see Section 6).

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 4. Modeling — multiple predictor types  *(≥2 types → avoids −3)*

> **Steps for this section (temporary — remove before submission)**
> - Compare at least two predictor families and present a results table (model × CV ROC-AUC × train time): LightGBM (primary); XGBoost and/or RandomForest; optional logistic-regression baseline, small MLP/1D-CNN over the weekly sequence.
> - Explain the long/stacked (Run No, unit, hour) framing and alternatives considered.
> - **Hyperparameter choices:** show how final values were obtained (CV / grid / Optuna). The short notebooks hard-code these values; this section is the required justification.

### 4.1 Framing and alternatives
We use the long (case, unit, hour) framing from Section 3.1 with a single classifier. We considered two alternatives:
- **one model per unit** (14 models): each model sees far less data, and the units share most of the structure (price, water value), which a unit-id feature captures;
- **a sequence model** (1D-CNN/RNN over the 168 hours): it could learn block structure directly, but the forward/backward price windows and LP features already give the tree models that context, at a much lower cost.

We did not pursue either, given the time available.

### 4.2 Predictors
| Model | Features | Time-split val. AUC | Notes |
|---|---|---|---|
| LP margin alone (no learning) | 1 | 0.956 (train set) | physical baseline |
| Logistic regression | all | `[TODO]` | linear baseline |
| Random forest | all | `[TODO]` | |
| XGBoost (Short notebook 2) | 73 (no LP) | 0.963 | best iteration 362 |
| **LightGBM (Short notebook 1)** | 81 (with LP) | **0.990** | best iteration 484, our best submission |

Validation uses the last ~15 % of training weeks (Section 5). **Caveat:** the two short notebooks currently differ in *both* model and feature set, so the 0.963 vs 0.990 gap mainly reflects the LP features, not XGBoost vs LightGBM. `[TODO: rerun XGBoost with the LP features for a fair comparison]`

### 4.3 Hyperparameters
- **LightGBM:** `learning_rate=0.05`, `num_leaves=255`, `min_child_samples=200`, `feature_fraction=0.8`, `bagging_fraction=0.8`.
- **XGBoost:** `max_depth=8` (similar capacity to 255 leaves), `min_child_weight=50`, `subsample` and `colsample_bytree` 0.8.
- **Number of rounds:** chosen by early stopping (100 rounds of patience) on the time-split validation set. The final model is refit on all training data with that number of rounds.
- **Why these values:** the large training set (6.9 M rows) supports high tree capacity, and row/column subsampling plus `min_child_samples` provide regularisation. In the earlier ablation, a more regularised setting (`num_leaves=63`, `min_child_samples=1000`, `feature_fraction=0.5`, L2 = 10) gave a mean of 0.9515 against 0.9503 for the baseline, again within seed noise, so we kept the simpler setting.
- `[TODO: describe the actual search used to pick these values, e.g. grid over num_leaves × min_child_samples]`

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 5. Validation strategy

Time-based split (early Run No → train, latest pre-test cases → validation) to mirror the 2015–2022 → 2023–2024 shift; why random K-fold leaks across adjacent weeks; the public/private LB gap and why we select final submissions on CV, not public rank.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 6. Model Interpretation  *(avoids −3)*

Show at least one (we recommend several):
- LightGBM gain **feature importance**
- **SHAP** summary plot
- **Partial dependence** of P(ON) on price-rank and water-value spread.
Narrative: confirm the model learned the economic rule (higher relative price ⇒ higher P(ON)).

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 7. Results & leaderboard

Final CV scores, public & private LB scores, the two selected submissions, and how many Virtual Teams we beat. Include the comparison table and any ablation results.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 8. Conclusion, limitations & reproducibility

What worked, what didn't (report negative results too — the brief rewards showing all models tried), limitations, and a pointer: *Short notebook 1 reproduces the best score; Short notebook 2 reproduces the second selected submission.*

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).
